In [2]:
!{sys.executable} -m pip install transformers
!{sys.executable} -m pip install torch tensorflow
!{sys.executable} -m pip install torch

Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached huggingface_hub-0.32.4-py3-none-any.whl.metadata (14 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-win_amd64.whl.metadata (2.1 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)
Using cached huggingface_hub-0.32.4-py3-none-any.whl (512 kB)
Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl (2.4 MB)
Using cached PyYAML-6.0.2-cp312-cp312-win_amd64.whl (156 kB)
Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl (308 kB)

   ---------- ----------------------------- 2/8 [pyyaml]
   --------------- ------------------------ 3/8 [fsspec]
   --------------- ------------------------ 3/8 [fsspec]
   ------------------------- -------------- 5/8 [huggingface-hub]
   -------

In [4]:
import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification,BertForMaskedLM

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') #tokenizer 会把句子转成模型可以理解的数字。
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
text = ("Napoleon revolutionised military organisation"
"Napoleon has legacy"
"Legacy still lives"
)
rep = tokenizer(text, return_tensors = "pt") #文本转换成 token IDs，并转成 PyTorch 的张量格式
print("Before Masking",rep.input_ids)
rand = torch.rand(rep.input_ids.shape)
mask_arr = (rand < 0.15) * (rep.input_ids != 101) * (rep.input_ids != 102)
# 注意:我们积极避免掩盖对应于101([Cls])和102([sep])的令牌。
selection = torch.flatten(mask_arr[0].nonzero()).tolist()
rep.input_ids[0, selection] = 103
after_masking = rep.input_ids
print("After Masking", after_masking)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Before Masking tensor([[  101,  8891,  4329,  5084,  2510,  5502,  2532, 15049,  2239,  2038,
          8027, 23115, 15719,  2145,  3268,   102]])
After Masking tensor([[  101,  8891,  4329,  5084,   103,  5502,  2532,   103,  2239,  2038,
          8027, 23115,   103,  2145,  3268,   102]])


In [7]:
outputs = model(**rep)
print(outputs)

NextSentencePredictorOutput(loss=None, logits=tensor([[ 5.7820, -5.2681]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [5]:
from transformers import BertTokenizer, BertForNextSentencePrediction #导入 NSP 模型和工具。
import torch
# torch 是PyTorch框架中用于做‘数学+神经网络’的核心引擎，可以构建张量（tensor）,训练神经网络，在GPU上加速，做自动求导和优化。

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') #同样加载预训练模型（这次是 NSP 版本）
model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased')
# 准备了两个句子，模型将判断 B 是不是 A 的下一句。
SentenceA = ("Napoleon has legacy")
SentenceB = ("Legacy still lives")
rep = tokenizer(SentenceA,SentenceB, return_tensors = "pt") #把两个句子拼接成 [CLS] SentenceA [SEP] SentenceB [SEP] 的结构，并转成 token ID。
print(rep) #打印出转换后的 token ID、token type（句子A是0，句子B是1）

{'input_ids': tensor([[ 101, 8891, 2038, 8027,  102, 8027, 2145, 3268,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [8]:
outputs = model(**rep)
logits = outputs.logits

# 结果是两个数：[IsNext, NotNext]
print("Logits:", logits)

# 看哪个更大（B 是不是 A 的下一句）
predicted = torch.argmax(logits).item()
if predicted == 0:
    print("✅ B 是 A 的下一句")
else:
    print("❌ B 不是 A 的下一句")


Logits: tensor([[ 5.7820, -5.2681]], grad_fn=<AddmmBackward0>)
✅ B 是 A 的下一句


# 🧪 使用 PyTorch 微调 BERT 进行文本分类

In [11]:
!{sys.executable} -m pip install transformers
!{sys.executable} -m pip install datasets

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   -- ------------------------------------- 1.6/25.7 MB 7.6 MB/s eta 0:00:04
   ------ --------------------------------- 3.9/25.7 MB 9.8 MB/s eta 0:00:03
   ---------- ----------------------------- 6.6/25.7 MB 10.3 MB/s eta 0:00:02
   ------------- -------------------------- 8.9/25.7 MB 10.9 MB/s eta 0:00:02
   ----------------- ---------------------- 11.5/25.7 MB 10.9 MB/s eta 0:00:02
   --------------------- ------------------ 13.9/25.7 MB 11.2 MB/s eta 0:00:02
   ------------------------- -------------- 16.5/25.7 MB 11.3 MB/s eta 0:00:01
   ----------------------------- ---------- 19.1/25.7 MB 11.3 MB/s eta 0:00:01
   --------------------------------- -

In [12]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

from torch.utils.data import DataLoader
from datasets import load_dataset

3. 加载数据集
我们使用 Hugging Face 的 datasets 库来加载一个示例数据集，例如 IMDb 电影评论数据集：

In [13]:
dataset = load_dataset('imdb')

C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\owner\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 374137.33 examples/s]


In [14]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

4. 数据预处理

In [15]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 50000/50000 [02:45<00:00, 302.49 examples/s]


5. 创建数据加载器

In [16]:
train_dataset = tokenized_datasets['train'].shuffle(seed=42).select(range(1000))  # 使用部分数据进行示例
eval_dataset = tokenized_datasets['test'].shuffle(seed=42).select(range(1000))

train_loader = DataLoader(train_dataset, batch_size=8)
eval_loader = DataLoader(eval_dataset, batch_size=8)

6. 加载预训练的 BERT 模型

In [18]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


7. 设置优化器

In [19]:
optimizer = AdamW(model.parameters(), lr=5e-5)

8. 训练模型

In [22]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    # 将列表中的字典合并为一个字典
    return {
        'input_ids': torch.tensor([item['input_ids'] for item in batch]),
        'attention_mask': torch.tensor([item['attention_mask'] for item in batch]),
        'labels': torch.tensor([item['label'] for item in batch])  # 注意label vs labels命名
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_dataset, batch_size=8, collate_fn=collate_fn)


In [23]:
model.train()
for batch in train_loader:
    inputs = {
        'input_ids': batch['input_ids'],
        'attention_mask': batch['attention_mask'],
        'labels': batch['labels']
    }
    outputs = model(**inputs)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

9. 评估模型

In [24]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in eval_loader:
        inputs = {key: val for key, val in batch.items() if key in ['input_ids', 'attention_mask']}
        labels = batch['labels']
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f'Accuracy: {correct / total}')

Accuracy: 0.774
